In [1]:

import numpy as np
import pickle
from collections import Counter

from qiskit import transpile, QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeFez

from qopt_best_practices.sat_mapping import SATMapper

from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples, evaluate_sparse_pauli


In [2]:
backend_options = dict(
    method='matrix_product_state',
    # method='statevector',
    matrix_product_state_max_bond_dimension='20', 
    device='CPU',
    precision='single',
    basis_gates = ['rx', 'ry', 'rz', 'cx']
)
# fake_fez = FakeFez()
backend = AerSimulator(**backend_options)

In [3]:
sampler = Sampler.from_backend(backend)

In [4]:
filename = 'test_N2_W2'
N = 2
T = 2
p: int = 6
shots = 2000

rng = np.random.default_rng()

data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

In [5]:
delta_beta, delta_gamma = 0.17, 0.17
betas = [(1-k/p) * delta_beta for k in range(p)]
gammas = [(k+1) / p * delta_gamma for k in range(p)]
fixed_params = betas + gammas

In [6]:
fixed_params

[0.17,
 0.1416666666666667,
 0.11333333333333336,
 0.085,
 0.05666666666666668,
 0.02833333333333333,
 0.028333333333333335,
 0.05666666666666667,
 0.085,
 0.11333333333333334,
 0.1416666666666667,
 0.17]

In [7]:
phis = ParameterVector('ϕ', num_qubits)
betas = ParameterVector('β', p)


init = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init.ry(phis[i], i)
    
mixer = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    mixer.ry(-phis[i], i)
    mixer.rz(-2*betas[0], i)
    mixer.ry(phis[i], i)
    
qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = p,
    flatten=True
)


In [8]:
graph = circuit_to_graph(qc, qc.parameters[p])

swap_strat = SwapStrategy.from_line(range(graph.order()))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

# remapped_g, sat_map, min_sat_layers = SATMapper(timeout=1).remap_graph_with_sat(
#     graph=graph, swap_strategy=swap_strat
# )
# if remapped_g is None:
#     raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(graph)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

circ_dict = circuit_construction(singles, doubles, None, swap_strat, edge_coloring, {}, p, init, mixer)

circuit = circ_dict["circuit_to_sample"]

In [9]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(β[2]), ParameterVectorElement(β[3]), ParameterVectorElement(β[4]), ParameterVectorElement(β[5]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1]), ParameterVectorElement(γ[2]), ParameterVectorElement(γ[3]), ParameterVectorElement(γ[4]), ParameterVectorElement(γ[5]), ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3]), ParameterVectorElement(ϕ[4]), ParameterVectorElement(ϕ[5]), ParameterVectorElement(ϕ[6]), ParameterVectorElement(ϕ[7])])

In [10]:
fixed_param_bind = {circuit.parameters[i]: fixed_params[i] for i in range(2*p)}
fixed_qc = circuit.assign_parameters(fixed_param_bind)

In [11]:
fixed_qc.draw(fold=-1)

┌──────────┐┌─────────────┐                                              ┌───┐                                                        ┌───┐                                                          ┌───┐                                                              ┌───────────┐┌───────────┐┌──────────┐                                                              ┌───┐                                                         ┌───┐                                                        ┌───┐                     ┌────────────┐ ┌───────────┐ ┌──────────────┐  ┌──────────┐ ┌─────────────┐                                                                                   ┌───┐                                                  ┌───┐                                                     ┌───┐                                                        ┌───────────┐┌──────────────┐┌──────────┐                                                            ┌───┐                                                        ┌───┐                                                      ┌───┐                    ┌───────────┐ ┌───────────┐ ┌───────────┐  ┌──────────┐ ┌─────────────┐                                                                                   ┌───┐                                                      ┌───┐                                                        ┌───┐                                                            ┌───────────┐┌──────────────┐┌──────────┐                                                      ┌───┐                                                   ┌───┐                                                ┌───┐                 ┌────────────┐┌───────────┐┌───────────────┐ ┌──────────┐              ┌─┐                                                                      
q_0: ┤ Ry(ϕ[0]) ├┤ Rz(-0.5525) ├────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■──────────────────┤ X ├──■───────────────────────────────────■───────────────────┤ X ├──■───────────────────────────────────■────────────────────■──┤ Ry(-ϕ[0]) ├┤ Rz(-0.34) ├┤ Ry(ϕ[0]) ├──■────────────────────■───────────────────────────────────■──┤ X ├──────────────────■───────────────────────────────────■──┤ X ├──────────────────■──────────────────────────────────■──┤ X ├─────────────────■───┤ Rz(-1.105) ├─┤ Ry(-ϕ[0]) ├─┤ Rz(-0.28333) ├──┤ Ry(ϕ[0]) ├─┤ Rz(-1.6575) ├───────────────────────────────────────────────────────────────────■───────────────┤ X ├──■───────────────────────────────■───────────────┤ X ├──■────────────────────────────────■─────────────────┤ X ├──■────────────────────────────────■─────────────────■──┤ Ry(-ϕ[0]) ├┤ Rz(-0.22667) ├┤ Ry(ϕ[0]) ├──■───────────────────■──────────────────────────────────■──┤ X ├──────────────────■──────────────────────────────────■──┤ X ├─────────────────■─────────────────────────────────■──┤ X ├────────────────■───┤ Rz(-2.21) ├─┤ Ry(-ϕ[0]) ├─┤ Rz(-0.17) ├──┤ Ry(ϕ[0]) ├─┤ Rz(-2.7625) ├──────────────────────────────────────────────────────────────────■────────────────┤ X ├──■─────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■──────────────────┤ X ├──■──────────────────────────────────■───────────────────■──┤ Ry(-ϕ[0]) ├┤ Rz(-0.11333) ├┤ Ry(ϕ[0]) ├──■────────────────■───────────────────────────────■──┤ X ├────────────────■───────────────────────────────■──┤ X ├──────────────■──────────────────────────────■──┤ X ├──────────────■──┤ Rz(-3.315) ├┤ Ry(-ϕ[0]) ├┤ Rz(-0.056667) ├─┤ Ry(ϕ[0]) ├──────────────┤M├──────────────────────────────────────────────────────────────────────
     ├──────────┤├─────────────┴┐                         ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐ └─┬─┘┌─┴─┐                     ┌───┐     ┌─┴─┐┌───────────────┐└─┬─┘┌─┴─┐                     ┌───┐     ┌─┴─┐┌─────────────┐ ┌─┴─┐├───────────┤├───────────┤├──────────┤┌─┴─┐┌─────────────┐ ┌─┴─┐     ┌───┐                     ┌─┴─┐└─┬─┘┌──────────────┐┌─┴

In [26]:
eta = 1
eps = 0.005

def normalisation(energies):
    return sum([np.exp(-beta_T * E) for E in energies])

def boltzmann(energies, Z):
    return np.exp(- beta_T * energies) / Z

def bias(boltzmanns, q, samples):
    return sum([boltzmanns[i] * (-1 if samples[i][q] == '1' else 1) for i in range(len(samples))])

def get_biases(samples, energies):
    Z = normalisation(energies)
    boltzmanns = boltzmann(energies, Z)
    return np.array([bias(boltzmanns, q, samples) for q in range(num_qubits)][::-1])

def get_angles(sample_energy_count: list[tuple[str, float, int]]):
    energies = [sample_energy_count[idx][2] * [sample_energy_count[idx][1]] for idx in range(len(sample_energy_count))]
    flat_energies = np.array([x for xs in energies for x in xs])
    samples = [sample_energy_count[idx][2] * [sample_energy_count[idx][0]] for idx in range(len(sample_energy_count))]
    flat_samples = [x for xs in samples for x in xs]
    biases = get_biases(flat_samples, flat_energies)
    # print('Biases')
    # print(np.round(biases, 2))
    probabilities = 0.5 * (1 - eta * biases)
    probabilities[probabilities < eps] = eps
    probabilities[probabilities > 1 - eps] = 1 - eps
    
    angles = 2 * np.arcsin(np.sqrt(probabilities))
    # print('Angles')
    # print(np.round(angles, 2))
    return angles

In [27]:
def iteration(angles):
    sample_circuit = fixed_qc.assign_parameters(angles, inplace=False)
    sampler_job = sampler.run([sample_circuit],shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    
    evals = evaluate_sparse_pauli_samples(counts.keys(), cost_op) + ising_offset
    energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
    flat_energies = [x for xs in energies for x in xs]
    total_energy = np.mean(flat_energies)
    
    sample_energy_count = [(sample, evals[idx], count) for idx, (sample, count) in enumerate(counts.items())]
    
    history.append([sample_energy_count, total_energy])
    new_angles = get_angles(sample_energy_count)
    return new_angles

In [30]:
# init_angles = np.pi/2 * np.ones((num_qubits,))
prob = 1 / (2 * N)
theta = 2 * np.arcsin(np.sqrt(prob))
init_angles = theta * np.ones((num_qubits,))
# init_angles = np.pi * np.array([1,0,0,0,0,0,1,0])
# init_angles[1] += 0.2
history = []
angles = [init_angles]
for i in range(10):
    beta_T = 0.1 * ((10**0.5 - 1) * i / 9 + 1) ** 2
    angles.append(iteration(angles[-1]))

In [31]:
for i in range(10):
    print(i)
    counter = Counter({history[i][0][j][0]: history[i][0][j][2] for j in range(len(history[i][0]))})
    energy_counter = Counter([x for xs in [[history[i][0][j][1]] * history[i][0][j][2] for j in range(len(history[i][0]))] for x in xs])
    energy = history[i][1]
    print(energy)
    print(counter.most_common(5))
    print(evaluate_sparse_pauli_samples([e[0] for e in counter.most_common(5)], hamiltonian) + ising_offset)
    
    print(energy_counter.most_common(5))
    print()

0
30.642
[('000100000000000000000000', 14), ('000010000000000000000000', 12), ('010000000000000000001000', 10), ('000000100000000000000010', 9), ('000000000001000000000100', 9)]
[50. 48. 39. 37. 37.]
[(np.float64(26.0), 435), (np.float64(37.0), 272), (np.float64(28.0), 225), (np.float64(21.0), 163), (np.float64(30.0), 131)]

1
30.117
[('000000000010000001000000', 23), ('010000000000000000000100', 17), ('000000000001000010000000', 12), ('000010000000000000000001', 11), ('000000000001000000000001', 11)]
[37. 39. 37. 37. 37.]
[(np.float64(26.0), 465), (np.float64(37.0), 399), (np.float64(28.0), 211), (np.float64(21.0), 176), (np.float64(41.0), 104)]

2
24.095
[('010000000010001000000010', 30), ('010000000010000010000001', 23), ('010000000010001000000001', 21), ('100000000010001000000010', 19), ('100000000010001000000001', 18)]
[ 5. 12.  5. 10. 10.]
[(np.float64(26.0), 438), (np.float64(17.0), 259), (np.float64(37.0), 221), (np.float64(21.0), 214), (np.float64(28.0), 201)]

3
17.24
[('0100